# GPU RAID — подготовка Kaggle Dataset с моделями

Один раз скачиваем модели в датасет — потом воркер стартует за ~3 минуты
без скачивания (датасеты монтируются в `/kaggle/input` мгновенно).

Раскладка датасета: подпапки с именами папок моделей ComfyUI
(`checkpoints/`, `vae/`, `text_encoders/`, `unet/`, `loras/`, …) + `manifest.json`.

⚠️ Лимит `/kaggle/working` — ~20 ГБ: собирайте датасеты порциями
(отдельно SDXL, отдельно Wan GGUF и т.д.).

In [ ]:
# ================= ЧТО КАЧАЕМ =================
# (repo_id, файл_в_репо, папка_моделей_ComfyUI)
# Проверяйте имена файлов на странице репозитория HF — они иногда меняются!
SPECS = [
    ("stabilityai/stable-diffusion-xl-base-1.0", "sd_xl_base_1.0.safetensors", "checkpoints"),
    # --- Wan 2.2 i2v в GGUF для T4 (раскомментируйте нужное): ---
    # ("QuantStack/Wan2.2-I2V-A14B-GGUF", "HighNoise/Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf", "unet"),
    # ("QuantStack/Wan2.2-I2V-A14B-GGUF", "LowNoise/Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf", "unet"),
    # ("city96/umt5-xxl-encoder-gguf", "umt5-xxl-encoder-Q5_K_M.gguf", "text_encoders"),
    # ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/vae/wan_2.1_vae.safetensors", "vae"),
    # --- MiniMax H3 (~40 ГБ суммарно!) в ОДИН датасет не влезет: лимит
    # /kaggle/working ~20 ГБ. Либо собирайте в два захода (два датасета:
    # отдельно diffusion_models, отдельно text_encoders+vae), либо вообще
    # без датасета — в ноутбуке воркера MODEL_PRESET="minimax_h3" качает
    # напрямую с HF за ~5–8 минут (через HF-кэш, лимит не задевается):
    # ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
    # ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    # ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    # ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
]

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print(f"{len(SPECS)} файлов к скачиванию")

In [ ]:
# ================= СКАЧИВАНИЕ + MANIFEST =================
import json, os, shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub[hf_transfer]"])
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

OUT = "/kaggle/working/gpuraid_dataset"
manifest = {"files": []}
for repo_id, filename, folder in SPECS:
    name = os.path.basename(filename)
    dst_dir = os.path.join(OUT, folder)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, name)
    if not os.path.exists(dst):
        print(f"скачиваю {repo_id}/{filename} …")
        path = hf_hub_download(repo_id=repo_id, filename=filename, token=HF_TOKEN or None)
        shutil.copy(path, dst)
    manifest["files"].append({"name": name, "path": f"{folder}/{name}", "model_folder": folder})
    print(f"  ok: {folder}/{name} ({os.path.getsize(dst) // (1 << 20)} МБ)")

with open(os.path.join(OUT, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=1)
print("\nГотово:", OUT)

### Как сохранить как Dataset

1. **Save Version** (справа вверху) → дождитесь завершения.
2. Откройте сохранённую версию → вкладка **Output** → у папки
   `gpuraid_dataset` нажмите **⋮ → New Dataset** → назовите, например,
   `gpuraid-sdxl` (private).
3. В ноутбуке воркера: **Add Input** → выберите этот датасет.

Рекомендуемые датасеты под вашу библиотеку:
- `gpuraid-sdxl` — SDXL-чекпоинт (~7 ГБ) для страйпинга картинок;
- `gpuraid-wan-gguf` — Wan 2.2 i2v Q4 GGUF + UMT5 GGUF + VAE (~12 ГБ) для offload видео;
- Flux на T4 не рекомендуется — для него берите Colab L4.